# Stage 5: spatial correction audit

This self-contained notebook evaluates a small nearest-well residual correction. It creates missing Stage 2--4 prerequisites automatically. Stage 5 is promoted only if normal well folds, leave-spatial-block-out folds, coordinate shuffle, bootstrap, and tail gates all pass.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess

REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
data_dir = Path('/content/rogii-data')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir = drive_root / 'artifacts'
if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
drive_data = drive_root / 'data'
assert (drive_data / 'train').is_dir(), f'Missing train directory: {drive_data / "train"}'
if not (data_dir / 'train').is_dir():
    shutil.copytree(drive_data, data_dir, dirs_exist_ok=True)
artifact_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
stage2_dir = artifact_dir / 'stage2_pf_trend_blend_full_v001'
if not (stage2_dir / 'metrics.json').is_file():
    assert not stage2_dir.exists() or not any(stage2_dir.iterdir()), f'Incomplete run: {stage2_dir}'
    subprocess.run(['uv', 'run', 'rogii-train', '--config', 'configs/experiment/pf_trend_blend.yaml', '--run-id', stage2_dir.name, '--data-dir', str(data_dir), '--artifact-dir', str(artifact_dir)], cwd=repo_dir, check=True)
stage3_dir = artifact_dir / 'stage3_residual_hgb_full_v001'
if not (stage3_dir / 'metrics.json').is_file():
    assert not stage3_dir.exists() or not any(stage3_dir.iterdir()), f'Incomplete run: {stage3_dir}'
    subprocess.run(['uv', 'run', 'rogii-residual', '--config', 'configs/experiment/stage3_residual_hgb.yaml', '--base-run', str(stage2_dir), '--run-id', stage3_dir.name, '--data-dir', str(data_dir), '--artifact-dir', str(artifact_dir)], cwd=repo_dir, check=True)
stage4_dir = artifact_dir / 'stage4_tail_path_full_v001'
if not (stage4_dir / 'metrics.json').is_file():
    assert not stage4_dir.exists() or not any(stage4_dir.iterdir()), f'Incomplete run: {stage4_dir}'
    subprocess.run(['uv', 'run', 'rogii-stage4', '--config', 'configs/experiment/stage4_tail_path.yaml', '--base-run', str(stage3_dir), '--run-id', stage4_dir.name, '--data-dir', str(data_dir), '--artifact-dir', str(artifact_dir)], cwd=repo_dir, check=True)
print('Stage 4 RMSE:', json.loads((stage4_dir / 'metrics.json').read_text())['pooled_rmse'])

In [ ]:
SMOKE_RUN_ID = 'stage5_spatial_audit_smoke_v001'
smoke_dir = artifact_dir / SMOKE_RUN_ID
if not (smoke_dir / 'spatial_audit.json').is_file():
    subprocess.run(['uv', 'run', 'rogii-spatial', '--config', 'configs/experiment/stage5_spatial_knn.yaml', '--base-run', str(stage4_dir), '--run-id', SMOKE_RUN_ID, '--data-dir', str(data_dir), '--artifact-dir', str(artifact_dir), '--limit-wells', '8'], cwd=repo_dir, check=True)
print('Smoke test completed; its score is not used for spatial selection.')

In [ ]:
RUN_ID = 'stage5_spatial_audit_full_v001'
run_dir = artifact_dir / RUN_ID
if not (run_dir / 'spatial_audit.json').is_file():
    subprocess.run(['uv', 'run', 'rogii-spatial', '--config', 'configs/experiment/stage5_spatial_knn.yaml', '--base-run', str(stage4_dir), '--run-id', RUN_ID, '--data-dir', str(data_dir), '--artifact-dir', str(artifact_dir)], cwd=repo_dir, check=True)
audit = json.loads((run_dir / 'spatial_audit.json').read_text())
{key: audit[key] for key in ['base_pooled_rmse', 'standard_pooled_rmse', 'standard_delta', 'spatial_block_pooled_rmse', 'spatial_block_delta', 'shuffle_pooled_rmse', 'shuffle_delta', 'promoted']}

In [ ]:
audit['gates'], audit['standard_fold_deltas'], audit['spatial_block_fold_deltas'], audit['bootstrap']